<h1 style="text-align: center;">HEISENBERG-BASED FAULT LOCALIZATION (HBFL)</h1>

<h2 style="text-align: center;">Download Initial Dependencies</h2>

In [6]:
import numpy as np
import pandas as pd
import json
import math
from tqdm import tqdm
from qiskit.quantum_info import SparsePauliOp, Operator, Pauli, Clifford
import qiskit.qasm2 as q2
from qiskit import QuantumCircuit
from IPython.display import display

from heisenbergUtilities import *

In [7]:
# from qiskit_aer import AerSimulator
# simulator = AerSimulator()

# print(simulator.available_devices())

<h3>Download MQT Benchmark circuits.</h3>

We'll run this experiment on 11 different quantum circuits, including DJ, GHZ, GraphState, Grover No-Ancilla, QAOA, QFT, QGAN, QPE Exact, QPE Inexact, RealAmpRandom, and VQE.



<h3>Generate mutants.</h3>

We'll start with mutants that add an unnecessary gate, then add mutants that replace a certain gate later. Missing code in SBFL is still a more recent topic even in classical programs.

The mutants were downloaded from QMutBench, choosing specifically mutants that added a gate at any location with 0-10% survival scores

<h2 style="text-align: center;">CONTROL PANEL</h2>

In [8]:
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
#-------------------------------------------------------------------------GENERAL PARAMETERS---------------------------------------------------------------------------#
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
ALG_NAME = "qpeinexact"
ORIGINAL_QASM = f"{ALG_NAME}_5_qubits.qasm"                                                 #| The original algorithm QASM circuit file name.                           
ORIGINAL_QASM_FOLDER = "SBFL_circuits/MQTBench/"                                            #| The folder containing the original QASM circuit file.                    
QASM_MUTANT_FOLDER = f"SBFL_circuits/QMutBenchMutants/Mutants_{ALG_NAME}_5_qubits/"         #| The folder containing all mutants related to the original QASM circuit.  
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
#-------------------------------------------------------------------------SB-QOPS PARAMETERS---------------------------------------------------------------------------#
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
RUNS = 10                                                                                   #| Number of times SB-QOPS will run to find a failing or passing test case.            
SHOTS = 20000                                                                               #| Number of shots for SB-QOPS to use to create the compact program specification.     
EPOCH = 80                                                                                  #| Number of epochs SB-QOPS will run to navigate the search space of test cases.       
SBQOPS_TOL = 0.1                                                                            #| Tolerance value SB-QOPS uses to determine if a test case is failing or passing.     
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
#---------------------------------------------------------------------HEISENBERG SBFL PARAMETERS-----------------------------------------------------------------------#
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
LAMBDA_WEIGHT = 1                                                                           #| The amount to adjust the "weight difference" in the heuristic           #
LAMBDA_COMP = 1                                                                             #| The amount to weigh the "composition difference" in the heuristic       #
LAMBDA_SUPPORT = 1                                                                          #| The amount to weigh the "support overlap" in the heuristic              #
LAMBDA_DIAMETER = 1                                                                         #| The amount to weigh the "diameter difference" in the heuristic          #
LAMBDA_PHASE = 1                                                                            #| The amount to weigh the "phase" in the heuristic                        #
CUSTOM_WEIGHT = 0.25                                                                        #| The amount of weight the "dstar boost" has in the custom heuristic (WIP)#
ATOL = 1e-4                                                                                 #| The tolerance value used to prune negligible magnitudes during Pauli propagation.   
MAX_TERMS = 100                                                                             #| The maximum number of terms to keep during Pauli propagation. If None, use all.     
SEARCH_STEP = 3                                                                             #| The search step size used during Pauli propagation. If None, pauli-prop uses 4.     
VERBOSE = 1                                                                                 #| A boolean value to determine if the program should print out detailed information.  
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#


Generate the linked list of all gates found in a given circuit

In [9]:
"""
LinkedListNode: A class that holds information related to a gate in a quantum circuit. 

PROPERTIES:
    - value (String): The name of the quantum gate
    - depth (int): The depth of the gate in the circuit
    - count (Int): The probability that the gate will meaningfully change the Pauli string.
    - next (LinkedListNode): The next gate in the circuit

"""

class LinkedListNode:
    def __init__(self, name="Initial", depth=0, count=0,  next=None):
        self.value = name
        self.depth = depth
        self.next = next
        self.count = count

class LinkedList:
    def __init__(self):
        self.head = None

    def append(self, name, depth):
        new_node = LinkedListNode(name, depth)
        if not self.head:
            self.head = new_node
            return
        last_node = self.head
        while last_node.next:
            last_node = last_node.next
        last_node.next = new_node

    def mark(self, depth):
        current_node = self.head
        while current_node:
            if current_node.depth == depth:
                current_node.count += 1
                return
            current_node = current_node.next

    def reset(self):
        current_node = self.head
        while current_node:
            current_node.count = 0
            current_node = current_node.next

<h3>Load correct circuit</h3>
Here we want to load the correct version of the circuit from MQT. We use this both to generate a PS_compact from/for SB-QOPS and to check results from SBFL.

In [10]:
correct_circuit = load_program(ORIGINAL_QASM, ORIGINAL_QASM_FOLDER).copy()
correct_list = LinkedList()
correct_list = construct_list(correct_list, correct_circuit, inverse=False)

<h2 style="text-align: center;"> SB-QOPS </h2>

This is where we will use SB-QOPS on a provided circuit and its mutants

For a single mutant, it took about 2 minutes to generate a 20 test suite (10 fail, 10 pass) on an RTX 4090 SUPER. It can only be run on a Linux environment since the AER Simulator does not support Windows

Expect this cell to take up to a few hours, BUT this cell should only need to be run once per algorithm with mutants under test. After it saves to the csv file at the end, this cell can be commented out as to not accidentally run it again. 

In the future if we want to test this methodology on higher qubit circuits or other algorithms, we'll likely either want to reduce the number of tests in the suite or the number of mutants we're analyzing

In [11]:
# import QOPS as qops
# from QOPS_test import get_compact_program_specification_Z
# from pathlib import Path

# ga_result = pd.DataFrame(columns=['Program',"path_to_mutant",'catch_avg','avg_fitness','failing_testcases', 'passing_testcases'])
# program_history = {}

# #program_specification = The compact program specification. SB-QOPS needs this to generate failing test cases.
# # In a public-use environment, this would be provided by the user.
# program_specification = get_compact_program_specification_Z(correct_circuit, shots=SHOTS)

# #mutants = A python list of qiskit QuantumCircuit variables with a mutation in them
# mutants = []
# mutants_names = []
# root = Path(QASM_MUTANT_FOLDER)
# for qasm_mutant in root.rglob("*.qasm"):
#     mutants.append(load_program(qasm_mutant.name, QASM_MUTANT_FOLDER))
#     mutants_names.append(qasm_mutant.name)

# for mutant_index,mutant in enumerate(mutants):
#     tester = qops.Circuit_Tester(CUT=mutant)
#     tester.set_applicable_families_Z(program_specification)
#     mutants_per_run = []
#     fitness_per_run = []
#     failing_testcases_per_run = []
#     history_per_run = []

#     print("NOW RUNNING TO FIND FAILING TESTS")
#     #For a predefined number of attempts, try to find a set of failing test cases (output is above predefined tolerance)
#     for runs in range(RUNS):
#         killed = 0
#         pauli = {}
#         fitness = 0
#         for i in range(len(tester.applicable_families)):
#             best_function,testcase, history = tester.run_mealoneplusone(i, EPOCH)
#             if best_function > SBQOPS_TOL:
#                 killed = 1
#                 pauli = testcase
#                 fitness = best_function
#                 break
#         mutants_per_run.append(killed)
#         failing_testcases_per_run.append(pauli)
#         fitness_per_run.append(fitness)
#         history_per_run.append(history)

#     avg_score = np.mean(mutants_per_run)
#     avg_fitness = np.mean(fitness_per_run)

#     #Here is our new, modified algorithm from the SB-QOPS method that attempts to find passing test cases as well. We'll need them for SBFL
#     passing_testcases_per_run = []

#     print("NOW RUNNING TO FIND PASSING TESTS")
#     for runs in range(RUNS):
#         pauli = {}
#         fitness = 0
#         for i in range(len(tester.applicable_families)):
#             best_function, testcase, history = tester.run_reverse_mealoneplusone(i,EPOCH)
#             if best_function < SBQOPS_TOL:
#                 pauli = testcase
#                 break
#         passing_testcases_per_run.append(pauli)

#     #Replace static program file with "program_name" when ready to test fully
#     """
#     ga_result: A pandas dataframe with the following columns
#         Program: Name of the mutant quantum circuit file
#         Path_To_Mutant: Path to the mutant file
#         Catch_Avg: The average number of times the mutant was identified by SB-QOPS
#         Avg_Fitness: The average fitness of the search algorithm used. Higher usually indicates higher Catch_Avg
#         Failing_Testcases: A list of dicts indicating the set of Pauli strings and the weights that should be applied to generate a failing test case
#         Passing_Testcases: A list of dicts indicating the set of Pauli strings and the weights that should be applied to generate a passing test case
#     """
#     ga_result = pd.concat([pd.DataFrame([[mutants_names[mutant_index],QASM_MUTANT_FOLDER,avg_score,avg_fitness,json.dumps(failing_testcases_per_run), json.dumps(passing_testcases_per_run)]],columns=ga_result.columns),ga_result],ignore_index=True)
#     ga_result.to_csv(f'realamprandom_ga_result', index=False)

ga_result is a slightly altered pandas Dataframe with similar structure found in SB-QOPS. The differences are changing the column "Program" to be the name of the mutant file instead of the original quantum circuit, changing "Mutant" to be the path to the mutant file instead of being an arbitrary index value, and adding a passing_testcases column that returns Pauli strings and coefficients that generate passing tests.

Now we want to run SBFL using Heisenberg evolution trees on each circuit placed in ga_result

Evolution analysis can vary widely in runtime. Some of the simplest circuits take seconds to analyze and only took upwards of 3-4 minutes to get results on the whole dataset (GHZ was the fastest taking 3 minutes to run). Other, more complex algorithms can take upwards of 6-10 minutes PER MUTANT (RealAmpRandom took longest so far with 14 hours).

The runtime of this code segment is directly tied to the depth of the circuit being analyzed and the number of 2-branching gates such as rotation or phase gates that exist in the circuit.

This is something to note in the final paper. Regardless of accuracy this methodology takes a long time to run. If results are promising, then we'll want to look into the tradeoffs between EXAM and hyperparameters to speed this up. Primarily the atol, max_terms, and search_step parameters in the evolve_pauli_exact method in HeisenbergUtilities.py


<h2 style="text-align: center;">HEISENBERG EVOLUTIONS (PAULI PROPAGATION)</h2>

In [12]:
from heisenbergTree import *

LAMBDAS = [LAMBDA_WEIGHT, LAMBDA_COMP, LAMBDA_SUPPORT, LAMBDA_DIAMETER, LAMBDA_PHASE]
ga_result = pd.read_csv(f'{ALG_NAME}_ga_result.csv')
custom_sbfl_result = pd.DataFrame(columns=['Program','path_to_mutant','best_exam_score', 'worst_exam_score', 'avg_exam_score'])
barinel_sbfl_result = pd.DataFrame(columns=['Program','path_to_mutant','best_exam_score', 'worst_exam_score', 'avg_exam_score'])
dstar_sbfl_result = pd.DataFrame(columns=['Program','path_to_mutant','best_exam_score', 'worst_exam_score', 'avg_exam_score'])
mutant_num = 0

#For each mutant of a circuit, run the SBFL implementation
for mutant, path in zip(ga_result.loc[:,"Program"], ga_result.loc[:,"path_to_mutant"]):
    print("Now evolving the following mutant: ", mutant)

    #Extract the raw test cases found for each mutant
    raw_failing_testcases = format_test_case(ga_result, mutant, mutant_num, "failing_testcases")
    raw_passing_testcases = format_test_case(ga_result, mutant, mutant_num, "passing_testcases")

    circuit_inverse = load_program(mutant,path).copy().inverse()  # Invert to track backward evolution of the operator
    forward_mutant = load_program(mutant, path).copy()

    #Create the linked list of operations in the inverse circuit
    operation_list = LinkedList()
    operation_list = construct_list(operation_list, circuit_inverse, True)

    forward_list = LinkedList()
    forward_list = construct_list(forward_list, forward_mutant, False)

    #Create list of nodes in linked list. Somewhere down the road remove the linked list implementation. It's unnecessary and giving me a headache
    node_list = []
    node = operation_list.head
    while node:
        node_list.append(node)
        node = node.next

    #Perform Pauli propagation (Heisenberg evolution) for all test cases. Returns a dataframe with all counts for all operations
    global_frame_fail = heisenberg_evolve(circuit_inverse, 
                                          operation_list, 
                                          raw_failing_testcases, 
                                          "fail", 
                                          LAMBDAS,
                                          atol = ATOL, 
                                          max_terms = MAX_TERMS, 
                                          search_step = SEARCH_STEP)
    
    global_frame_pass = heisenberg_evolve(circuit_inverse, 
                                          operation_list, 
                                          raw_passing_testcases, 
                                          "pass", 
                                          LAMBDAS,
                                          atol = ATOL, 
                                          max_terms = MAX_TERMS, 
                                          search_step = SEARCH_STEP)

    global_frame = pd.concat([global_frame_fail, global_frame_pass], ignore_index=True)

    global_frame = global_frame.drop(["Initial"],axis=1)
    if VERBOSE:
        display(global_frame)

    custom_scores = custom_sbfl(global_frame, CUSTOM_WEIGHT)
    barinel_scores = barinel(global_frame)
    dstar_scores = dstar(global_frame)

    if VERBOSE:
        print("BARINEL SCORES")
        display(barinel_scores)
        print("custom SCORES")
        display(custom_scores)
        print("DSTAR SCORES")
        display(dstar_scores)
    
    # Here is where analysis of the methodology begins. 
    # We first extract the position of the erroneous gate by comparing it to the original MQT gate
    # NOTE: This method is intended for single-added gates for now. Other types of mutants will require other methods later
    error_gate_location = find_erroneous_gate(forward_mutant, correct_circuit)

    if VERBOSE:
        print("ERROR GATE LOCATION:")
        print(error_gate_location)

    #Calculate best, worst, and average EXAM for Barinel, custom, and DStar
    barinel_best_exam, barinel_worst_exam, barinel_average_exam = calc_exams(barinel_scores, error_gate_location, circuit_inverse)
    custom_best_exam, custom_worst_exam, custom_average_exam = calc_exams(custom_scores, error_gate_location, circuit_inverse)
    dstar_best_exam, dstar_worst_exam, dstar_average_exam = calc_exams(dstar_scores, error_gate_location, circuit_inverse)

    # Here we store the results from the HBFL analysis for both barinel and custom into a csv file for later analysis.
    barinel_sbfl_result = pd.concat([pd.DataFrame([[mutant,QASM_MUTANT_FOLDER,barinel_best_exam, barinel_worst_exam, barinel_average_exam]],columns=barinel_sbfl_result.columns),barinel_sbfl_result],ignore_index=True)
    barinel_sbfl_result.to_csv(f'{ALG_NAME}_barinel_sbfl_result.csv', index=False)

    custom_sbfl_result = pd.concat([pd.DataFrame([[mutant,QASM_MUTANT_FOLDER,custom_best_exam, custom_worst_exam, custom_average_exam]],columns=custom_sbfl_result.columns),custom_sbfl_result],ignore_index=True)
    custom_sbfl_result.to_csv(f'{ALG_NAME}_custom_sbfl_result.csv', index=False)

    dstar_sbfl_result = pd.concat([pd.DataFrame([[mutant,QASM_MUTANT_FOLDER,dstar_best_exam, dstar_worst_exam, dstar_average_exam]],columns=dstar_sbfl_result.columns),dstar_sbfl_result],ignore_index=True)
    dstar_sbfl_result.to_csv(f'{ALG_NAME}_dstar_sbfl_result.csv', index=False)

    mutant_num += 1

if VERBOSE:
    display(barinel_sbfl_result)
    display(custom_sbfl_result)
    

Now evolving the following mutant:  AddGate_ry_inGap_15_.qasm


100%|██████████| 10/10 [00:21<00:00,  2.15s/it]


,ry 21,h 20,cp 19,cp 18,cp 17,h 16,cp 15,cp 14,h 13,cp 12,...,swap 8,cp 7,cp 6,cp 5,x 4,h 3,h 2,h 1,h 0,pass/fail
0,0.541267,0.0,0.0,9.421020e-19,4.643985e-19,0.030795,0.103681,0.054003,0.045138,0.134841,...,0.120233,0.053619,0.012236,0.000189,0.121638,0.093051,0.091353,0.071782,0.067526,fail
1,0.700709,0.0,0.0,1.197371e-18,5.902306e-19,0.057530,0.183176,0.097800,0.072932,0.197564,...,0.173235,0.075493,0.021161,0.000276,0.187421,0.144844,0.133950,0.128439,0.090195,fail
2,0.568103,0.0,0.0,9.058335e-19,4.465203e-19,0.031047,0.103441,0.052640,0.053865,0.168088,...,0.162683,0.064967,0.010864,0.000188,0.152789,0.098585,0.113130,0.070132,0.070007,fail
3,0.618091,0.0,0.0,1.034266e-18,5.098297e-19,0.054182,0.179991,0.093401,0.065327,0.185360,...,0.159527,0.069989,0.019425,0.000240,0.162191,0.125566,0.124244,0.118851,0.085190,fail
4,0.528925,0.0,0.0,1.063958e-18,5.244661e-19,0.051063,0.171875,0.088320,0.068829,0.197762,...,0.170769,0.071493,0.017915,0.000245,0.162550,0.120767,0.125548,0.108999,0.072091,fail
5,0.567266,0.0,0.0,1.170214e-18,5.768436e-19,0.045040,0.154115,0.080101,0.065825,0.201150,...,0.182747,0.073189,0.016244,0.000246,0.161938,0.123282,0.124022,0.094663,0.064904,fail
6,0.369596,0.0,0.0,9.910915e-19,4.885473e-19,0.046026,0.157980,0.081079,0.060389,0.180477,...,0.174351,0.064816,0.015838,0.000224,0.140268,0.107214,0.107434,0.090960,0.037264,fail
7,0.603069,0.0,0.0,1.135911e-18,5.599343e-19,0.061702,0.208518,0.108751,0.078742,0.220927,...,0.169834,0.078391,0.020948,0.000248,0.187869,0.134235,0.138608,0.129648,0.084329,fail
8,0.397809,0.0,0.0,9.881090e-19,4.870771e-19,0.063272,0.209460,0.109054,0.067782,0.188121,...,0.170474,0.068372,0.021935,0.000243,0.151590,0.120017,0.114311,0.129724,0.049790,fail
9,0.395537,0.0,0.0,8.522161e-19,4.200902e-19,0.038363,0.138310,0.070337,0.051422,0.163157,...,0.138833,0.056809,0.013180,0.000152,0.132287,0.081499,0.096381,0.076770,0.048004,fail


[[1.32449911        nan        nan 1.1646367  1.1646367  1.32085555
  1.30055123 1.30527453 1.24937355 1.20236017 1.31600669 1.14119339
  1.24630704 1.12620186 1.15767737 1.29220254 1.22488457 1.20387325
  1.26054156 1.18571944 1.27184501 1.34760962]]
BARINEL SCORES


d:\GitHub\heisenberg-SBFL-experiment\heisenbergUtilities.py:326: RuntimeWarning: invalid value encountered in divide
  dstar_boost = max_fail.values/avg_fail.values


...

DSTAR SCORES


,ry 16,cp 12,cp 9,swap 10,cp 15,cp 20,swap 8,x 4,h 2,h 1,...,cp 14,h 11,h 13,cp 19,h 17,cp 7,h 21,cp 18,cp 6,cp 5
sum,0.962305,0.592222,0.560746,0.554737,0.54476,0.421787,0.300973,0.29538,0.201169,0.19081,...,0.162538,0.142824,0.117324,0.116225,0.087126,0.073709,0.04869,0.011171,0.006165,0.000376


ERROR GATE LOCATION:
16


,Program,path_to_mutant,best_exam_score,worst_exam_score,avg_exam_score
0,AddGate_ry_inGap_10_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeinex...,0.136364,0.136364,0.136364
1,AddGate_ry_inGap_16_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeinex...,0.181818,0.181818,0.181818
2,AddGate_rxx_inGap_16_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeinex...,0.045455,0.045455,0.045455
3,AddGate_rzz_inGap_10_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeinex...,0.590909,0.590909,0.590909
4,AddGate_h_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeinex...,0.090909,0.090909,0.090909
...,...,...,...,...,...
75,AddGate_h_inGap_11_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeinex...,0.045455,0.045455,0.045455
76,AddGate_rzz_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeinex...,0.590909,0.590909,0.590909
77,AddGate_y_inGap_10_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeinex...,0.136364,0.136364,0.136364
78,AddGate_y_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeinex...,0.136364,0.136364,0.136364


,Program,path_to_mutant,best_exam_score,worst_exam_score,avg_exam_score
0,AddGate_ry_inGap_10_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeinex...,0.090909,0.090909,0.090909
1,AddGate_ry_inGap_16_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeinex...,0.136364,0.136364,0.136364
2,AddGate_rxx_inGap_16_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeinex...,0.045455,0.045455,0.045455
3,AddGate_rzz_inGap_10_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeinex...,0.954545,0.954545,0.954545
4,AddGate_h_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeinex...,0.090909,0.090909,0.090909
...,...,...,...,...,...
75,AddGate_h_inGap_11_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeinex...,0.045455,0.045455,0.045455
76,AddGate_rzz_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeinex...,0.772727,0.772727,0.772727
77,AddGate_y_inGap_10_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeinex...,0.5,0.5,0.5
78,AddGate_y_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeinex...,0.181818,0.181818,0.181818
